# Meal Planning Agent — Prompt Experiments

Scratchpad for iterating on prompts before building the app.

## Initialization

In [8]:
from dotenv import load_dotenv
import requests
from agents import Agent, Runner, trace, function_tool, ModelSettings, OpenAIChatCompletionsModel
from agents.extensions.visualization import draw_graph
from openai import AsyncOpenAI
from pydantic import BaseModel, Field
import os
import asyncio

load_dotenv(override=True)

def assertKeyExists(key :str) -> str:
    value = os.getenv(key)
    assert value, f"{key} not set"
    return value

openai_api_key = assertKeyExists("OPENAI_API_KEY")
google_api_key = assertKeyExists("GOOGLE_API_KEY")
grok_api_key = assertKeyExists("GROK_API_KEY")

print("API keys loaded ✔")

API keys loaded ✔


In [ ]:
pushover_user = assertKeyExists("PUSHOVER_USER")
pushover_token = assertKeyExists("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"


def send_push_notification(message: str):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [3]:
high_effort_model = "gpt-5.6-sol"
default_model = "gpt-5.6-luna"
low_effort_model = "gpt-5.6-terra"

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.6-flash", openai_client=gemini_client)

GROK_BASE_URL = "https://api.x.ai/v1"
grok_client = AsyncOpenAI(base_url=GROK_BASE_URL, api_key=grok_api_key)
grok_model = OpenAIChatCompletionsModel(model="grok-4.5", openai_client=grok_client)


## Basic agent

A starting point with minimal agent to sanity-check the setup and test basic behavior.

In [ ]:
class PreparedDish(BaseModel):
    name: str = Field(description="The short name of a dish.")
    description: str = Field(description="A 1-2 sentence description of the dish.")
    special_diet_labels: str = Field(description="A list of any dietary restrictions that this meal satisfies. Examples: vegetarian, gluten-free, etc")


class PreparedDishes(BaseModel):
    dishes: list[PreparedDish] = Field(description="A list of ideas for prepared dishes.")

class MealPlanBrainstorm:
    entrees: list[PreparedDish]
    sides: list[PreparedDish] 

    def __init__(self, entrees: list[PreparedDish], sides: list[PreparedDish]) -> None:
        self.entrees = entrees
        self.sides = sides

    def __str__(self):
        return f"entrees: {self.entrees}, sides: {self.sides}"


instructions = '''
You are a meal planning assistant who helps people plan their meals for the week.
You are an expert on simple meals that require minimal amounts of preparation, reheat well, and are delicious.
'''

meal_planning_assistant = Agent(
    name="Meal Planner",
    instructions=instructions,
    model=default_model,
    output_type=PreparedDishes,
)
with trace("Basic Meal Plan"):
    meat_entrees = await Runner.run(meal_planning_assistant, "Suggest five different meat-based entrees.")
    vegetarian_entrees = await Runner.run(meal_planning_assistant, "Suggest five different meatless entrees.")
    sides = await Runner.run(meal_planning_assistant, "Suggest ten different meatless sides.")
    unfiltered_meal_plan = MealPlanBrainstorm(
        entrees = meat_entrees.final_output.dishes + vegetarian_entrees.final_output.dishes, 
        sides = sides.final_output.dishes,
    )
    print(unfiltered_meal_plan)

entrees: [PreparedDish(name='Slow Cooker Beef Chili', description='Ground beef simmered with beans, tomatoes, peppers, and warm spices. It reheats exceptionally well and can be served with rice, bread, or tortilla chips.', special_diet_labels='Gluten-free'), PreparedDish(name='Sheet-Pan Chicken Fajitas', description='Chicken strips roasted with bell peppers and onions, then served in tortillas or over rice. Prep is quick, and leftovers reheat easily.', special_diet_labels=''), PreparedDish(name='Turkey Meatballs with Marinara', description='Tender baked turkey meatballs simmered in marinara sauce, served with pasta, polenta, or crusty bread. Make a large batch for several meals.', special_diet_labels=''), PreparedDish(name='Pork Carnitas', description='Seasoned pork cooked until tender and shredded, then used in tacos, rice bowls, or salads. The meat keeps and reheats beautifully.', special_diet_labels='Gluten-free'), PreparedDish(name='Beef and Vegetable Stir-Fry', description='Thinly